# Clasificador de Desinformación sobre Inmigración
### BETO (BERT en español) — Fine-tuning mejorado en Google Colab con GPU

**Mejoras incorporadas:**
- Clasificación en **3 clases**: VERDADERO / CONTEXTO / FALSO
- **Early stopping** — para automáticamente si no mejora
- **Label smoothing** — mejora la calibración de confianzas
- **Cosine LR scheduler + warmup** — entrenamiento más estable
- Split **80/10/10** con conjunto de validación separado
- **MAX_LEN=512** — más contexto por noticia

> **Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU**

## 0. Verificar GPU e instalar dependencias

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"Dispositivo : {gpu.name}")
    print(f"VRAM        : {gpu.total_memory / 1e9:.1f} GB")
else:
    print("SIN GPU — activa la T4 en Entorno de ejecución > Cambiar tipo de entorno")

In [ ]:
!pip install -q "transformers[torch]>=4.40" accelerate scikit-learn openpyxl sentencepiece datasets

## 1. Configuración global

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import re
import gc
import torch
warnings.filterwarnings("ignore")

# Evitar warnings de paralelismo
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ALLOC_CONF"]    = "expandable_segments:True"

# ── Modelo ──
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"  # BETO: BERT entrenado en español

# ── Clases ──
LABEL2ID = {"VERDADERO": 0, "CONTEXTO": 1, "FALSO": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

# ── Hiperparámetros ──
MAX_LEN          = 512    # tokens por noticia (más contexto que antes)
BATCH_SIZE       = 16     # por GPU
EPOCHS           = 6      # early stopping para antes si no mejora
LR               = 2e-5
WARMUP_RATIO     = 0.1    # calentamiento del 10% de los steps
WEIGHT_DECAY     = 0.01
LABEL_SMOOTHING  = 0.1    # evita que el modelo sea demasiado confiado
SEED             = 42
UMBRAL_CONFIANZA = 0.60   # confianza mínima para aceptar predicción

torch.manual_seed(SEED)
np.random.seed(SEED)

n_gpus = torch.cuda.device_count()
print(f"Modelo          : {MODEL_NAME}")
print(f"Clases          : {list(LABEL2ID.keys())}")
print(f"GPUs activos    : {n_gpus} → batch efectivo: {BATCH_SIZE * max(n_gpus,1)}")
print(f"MAX_LEN         : {MAX_LEN} tokens")

## 2. Subir los datasets
> Sube los dos archivos:
> - `dataset_inmigracion_unificado_limpio_tema.xlsx`
> - `dataset_inmigracion_falso_contexto.xlsx`

In [ ]:
from google.colab import files
uploaded = files.upload()

## 3. Carga y preparación del dataset

In [ ]:
FILE_REAL = "dataset_inmigracion_unificado_limpio_tema.xlsx"
FILE_FAKE = "dataset_inmigracion_falso_contexto.xlsx"

df_real = pd.read_excel(FILE_REAL)
df_fake = pd.read_excel(FILE_FAKE)

print("── Dataset real ──")
print(df_real["etiqueta"].value_counts().to_string())
print(f"   Total: {len(df_real)}")
print()
print("── Dataset sintético (adulteraciones IA) ──")
print(df_fake["etiqueta"].value_counts().to_string())
print(f"   Total: {len(df_fake)}")

In [ ]:
# Dataset real: ALERTA (11 filas) → VERDADERO (son alertas verídicas)
df1 = df_real[["titulo", "texto", "etiqueta"]].copy()
df1["etiqueta"] = df1["etiqueta"].replace("ALERTA", "VERDADERO")
df1 = df1[df1["etiqueta"].isin(LABEL2ID)]

# Dataset sintético: FALSO y CONTEXTO
df2 = df_fake[["titulo", "texto", "etiqueta"]].copy()
df2 = df2[df2["etiqueta"].isin(LABEL2ID)]

# Combinar y limpiar
df = pd.concat([df1, df2], ignore_index=True)
df = df.dropna(subset=["titulo", "texto"])
df["titulo"] = df["titulo"].astype(str).str.strip()
df["texto"]  = df["texto"].astype(str).str.strip()
df = df[(df["titulo"].str.len() > 5) & (df["texto"].str.len() > 20)]
df["label"] = df["etiqueta"].map(LABEL2ID).astype(int)
df = df.reset_index(drop=True)

print("── Distribución final ──")
for etiq, cnt in df["etiqueta"].value_counts().items():
    bar = "█" * (cnt // 100)
    print(f"  {etiq:<12} {cnt:5d}  {cnt/len(df)*100:5.1f}%  {bar}")
print(f"\n  TOTAL        {len(df):5d}")

## 4. Split train / val / test (80% / 10% / 10%)
> Se usa un conjunto de **validación separado** para el early stopping,
> evitando que el test set influya en las decisiones de entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.10, random_state=SEED, stratify=df["label"]
)
train_df, val_df = train_test_split(
    train_df, test_size=0.111, random_state=SEED, stratify=train_df["label"]
)  # 0.111 ≈ 10% del total

print(f"{"Split":<8} {"N":>6}  {"VERDADERO":>10}  {"CONTEXTO":>9}  {"FALSO":>7}")
print("-" * 50)
for nombre, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    c = split["etiqueta"].value_counts()
    print(f"{nombre:<8} {len(split):>6}  "
          f"{c.get("VERDADERO", 0):>10}  "
          f"{c.get("CONTEXTO", 0):>9}  "
          f"{c.get("FALSO", 0):>7}")

## 5. Tokenizador y clase Dataset

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset as TorchDataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NoticiaDataset(TorchDataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.textos = (
            dataframe["titulo"].values + " " + dataframe["texto"].values
        ).tolist()
        self.labels    = dataframe["label"].values.tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.textos[idx],
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }
        if "token_type_ids" in enc:
            item["token_type_ids"] = enc["token_type_ids"].squeeze()
        return item

train_ds = NoticiaDataset(train_df, tokenizer, MAX_LEN)
val_ds   = NoticiaDataset(val_df,   tokenizer, MAX_LEN)
test_ds  = NoticiaDataset(test_df,  tokenizer, MAX_LEN)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Ejemplo — label: {ID2LABEL[train_ds[0]["labels"].item()]}")

## 6. Cargar modelo BETO

In [ ]:
from transformers import AutoModelForSequenceClassification

# Limpiar VRAM por si quedaron tensores de ejecuciones anteriores
if "model" in dir():
    del model
torch.cuda.empty_cache()
gc.collect()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Modelo     : {MODEL_NAME}")
print(f"Parámetros : {n_params:,}")
if torch.cuda.is_available():
    vram = torch.cuda.memory_allocated(0)
    print(f"VRAM usada : {vram / 1e9:.2f} GB")

## 7. Trainer con técnicas avanzadas
> - **Pesos de clase**: penaliza más los errores en clases minoritarias
> - **Label smoothing (α=0.1)**: en vez de [0,0,1] entrena contra [0.033, 0.033, 0.933] — reduce overconfidence
> - **Early stopping**: para automáticamente si el F1 no mejora en 3 épocas seguidas
> - **Cosine LR + warmup**: la tasa de aprendizaje sube suavemente al inicio y baja en coseno al final

In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import f1_score, accuracy_score, classification_report
from collections import Counter

# Pesos inversamente proporcionales a la frecuencia de cada clase
counts  = train_df["label"].value_counts().sort_index().values
weights = torch.tensor(
    len(train_df) / (len(LABEL2ID) * counts), dtype=torch.float
)
print("Pesos de clase:")
for i, w in enumerate(weights):
    print(f"  {ID2LABEL[i]:<12} {w:.4f}  (N={counts[i]})")
print(f"\nLabel smoothing  : {LABEL_SMOOTHING}")
print(f"Umbral confianza : {UMBRAL_CONFIANZA}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    dist  = Counter(preds.tolist())
    dist_str = "  ".join(f"{ID2LABEL[k]}:{v}" for k, v in sorted(dist.items()))
    print(f"  Distribución predicciones: {dist_str}")
    return {
        "accuracy"   : float(accuracy_score(labels, preds)),
        "f1_macro"   : float(f1_score(labels, preds, average="macro")),
        "f1_weighted": float(f1_score(labels, preds, average="weighted")),
        "f1_falso"   : float(f1_score(labels, preds, labels=[2], average="macro")),
    }

class WeightedTrainer(Trainer):
    """
    Pérdida = CrossEntropy ponderada + label smoothing.
    Label smoothing reduce overconfidence y mejora la calibración
    de probabilidades — las confianzas que devuelve el modelo son más reales.
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(
            weight=weights.to(outputs.logits.device),
            label_smoothing=LABEL_SMOOTHING,
        )
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

## 8. Entrenamiento

In [ ]:
args = TrainingArguments(
    output_dir                  = "./bert_checkpoints",
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 64,
    learning_rate               = LR,
    lr_scheduler_type           = "cosine",   # bajada suave al final
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 1,          # solo guarda el mejor checkpoint
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_falso",
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available(),
    dataloader_pin_memory       = True,
    logging_steps               = 50,
    report_to                   = "none",
    seed                        = SEED,
)

trainer = WeightedTrainer(
    model           = model,
    args            = args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

steps_epoca = len(train_ds) // (BATCH_SIZE * max(n_gpus, 1))
print(f"Steps por época : {steps_epoca}")
print(f"Épocas máx      : {EPOCHS}  (early stop si no mejora 3 épocas)")
print()
trainer.train()

## 9. Evaluación en test set

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

output = trainer.predict(test_ds)
y_pred = np.argmax(output.predictions, axis=-1)
y_true = output.label_ids

target_names = [ID2LABEL[i] for i in sorted(ID2LABEL)]

print("=" * 60)
print("RESULTADOS EN TEST SET")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=target_names, digits=3))

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.title("Matriz de Confusión — BETO Fine-tuning", fontweight="bold")
plt.ylabel("Real")
plt.xlabel("Predicho")
plt.tight_layout()
plt.savefig("evaluacion_beto.png", dpi=150)
plt.show()

## 10. Test de inferencia
> Prueba rápida con ejemplos reales para verificar que el modelo funciona correctamente.

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=MAX_LEN,
)

EMOJIS = {"VERDADERO": "🟢", "CONTEXTO": "🟡", "FALSO": "🔴"}

ejemplos = [
    "El Gobierno aprueba medidas para agilizar los expedientes de asilo en España",
    "Canarias registra un descenso del 85% en llegadas de cayucos respecto al año anterior",
    "Los inmigrantes ilegales cobran 4.700 euros al mes mientras los jubilados mueren de hambre",
    "El Gobierno prohíbe pedir antecedentes penales para facilitar la entrada de delincuentes",
    "Invasión planificada: el Gobierno traslada miles de jóvenes africanos sin ningún control",
    "Canarias acoge el triple de menores migrantes de lo que permite la ley",
]

print(f"{"Veredicto":<18} {"Conf":>6}  Titular")
print("-" * 80)
for texto in ejemplos:
    res   = pipe(texto)[0]
    emoji = EMOJIS.get(res["label"], "⚪")
    alerta = "⚠️" if res["score"] < UMBRAL_CONFIANZA else ""
    print(f"{emoji} {res["label"]:<14} {res["score"]:>5.1%} {alerta}  {texto[:55]}...")

## 11. Función predecir_noticia() — integración con el chatbot
> Esta es la función que llama el chatbot cuando recibe una noticia.
> Devuelve etiqueta, confianza, nivel de alerta y un `prompt_hint` listo para el LLM.

In [ ]:
def predecir_noticia(titulo: str, texto: str) -> dict:
    """
    Clasifica una noticia y devuelve la predicción lista para el chatbot.

    Args:
        titulo : Título de la noticia
        texto  : Cuerpo de la noticia

    Returns:
        dict con etiqueta, confianza, nivel_alerta y prompt_hint para el LLM
    """
    texto_input = (titulo + " " + texto).strip()

    res        = pipe(texto_input)[0]
    etiqueta   = res["label"]
    confianza  = res["score"]
    es_fiable  = confianza >= UMBRAL_CONFIANZA

    if etiqueta == "FALSO" and confianza >= 0.85:    nivel = "ALTO"
    elif etiqueta == "FALSO" and confianza >= 0.60:  nivel = "MEDIO"
    elif etiqueta == "CONTEXTO":                     nivel = "MEDIO"
    elif etiqueta == "VERDADERO" and confianza < UMBRAL_CONFIANZA: nivel = "REVISAR"
    else:                                            nivel = "BAJO"

    descripciones = {
        "VERDADERO": "PROBABLEMENTE VERDADERA",
        "CONTEXTO" : "DATOS REALES PRESENTADOS DE FORMA ENGAÑOSA",
        "FALSO"    : "PROBABLEMENTE FALSA O ENGAÑOSA",
    }

    aviso_confianza = "" if es_fiable else " (confianza baja — revisar con el RAG)"

    prompt_hint = (
        f"El clasificador BETO indica que esta noticia es "
        f"{descripciones.get(etiqueta, etiqueta)} "
        f"con una confianza del {confianza*100:.0f}%{aviso_confianza}. "
        f"Usa las evidencias del RAG para justificar la respuesta al usuario."
    )

    return {
        "etiqueta"    : etiqueta,
        "confianza"   : round(float(confianza), 3),
        "nivel_alerta": nivel,
        "es_fiable"   : es_fiable,
        "prompt_hint" : prompt_hint,
    }


# ── Demo ──────────────────────────────────────────────────────────────────
print("── DEMO predecir_noticia() ──\n")
casos = [
    ("Los inmigrantes colapsan los servicios de salud en España",
     "Según datos no verificados, los inmigrantes representan el 80% del gasto sanitario."),
    ("El Gobierno aprueba medidas de acogida para menores no acompañados",
     "El Consejo de Ministros ha aprobado hoy un plan de distribución de menores migrantes."),
]
for titulo, texto in casos:
    r = predecir_noticia(titulo, texto)
    print(f"Título        : {titulo[:60]}")
    for k, v in r.items():
        print(f"  {k:<14}: {v}")
    print()

## 12. Guardar y descargar el modelo

In [ ]:
MODEL_SAVE = "./modelo_beto_final"
os.makedirs(MODEL_SAVE, exist_ok=True)

trainer.save_model(MODEL_SAVE)
tokenizer.save_pretrained(MODEL_SAVE)

print(f"Modelo guardado en: {MODEL_SAVE}")
print("\nArchivos generados:")
for f in sorted(os.listdir(MODEL_SAVE)):
    size = os.path.getsize(os.path.join(MODEL_SAVE, f))
    size_str = f"{size/1e6:.1f} MB" if size > 1e5 else f"{size/1e3:.0f} KB"
    print(f"  {f:<40} {size_str}")

# Comprimir y descargar
!zip -r modelo_beto_final.zip ./modelo_beto_final

from google.colab import files
files.download("modelo_beto_final.zip")
files.download("evaluacion_beto.png")
print("\n✓ Descarga iniciada")